# ⚡ Laya Decision API on Kaggle T4 GPU (50-Call Benchmark)
### High-Throughput System 1 Non-Autoregressive Decision Engine (TypeSafe / Jev Compatible)

> ### 🚀 Quickstart in Kaggle:
> 1. **Accelerator:** Under **Notebook Options** in the right panel, set **GPU T4 x1** (or T4 x2).
> 2. **Internet:** Toggle **Internet ON** in the right panel.
> 3. **Run All:** Click **Run All** (or `Shift + Enter` through each cell).
>    - Port **8000** is used (Port 8888 is reserved by the Kaggle Jupyter server itself).
>    - No cloud account or external API key needed; runs in local **Keyless mode** directly on your T4 instance.

## 🔐 1. Environment & Keyless Mode Configuration
Configures local keyless environment variables on port `8000`.

In [8]:
import os
import sys

PORT = 8000

# Configure local keyless environment variables (Port 8000)
os.environ["UNSLOTH_API_KEY"] = "sk-local-kaggle-token"
os.environ["TYPESAFE_API_KEY"] = "sk-local-kaggle-token"
os.environ["TYPESAFE_BASE_URL"] = f"http://127.0.0.1:{PORT}"

# Laya server configurations
os.environ["LAYA_HOST"] = "127.0.0.1"
os.environ["LAYA_PORT"] = str(PORT)
os.environ["LAYA_MODELS"] = "multilingual"
os.environ["LAYA_DEVICE"] = "cuda"

# Safely attempt to load optional secrets from Kaggle if available
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    try:
        key = user_secrets.get_secret("UNSLOTH_API_KEY")
        os.environ["UNSLOTH_API_KEY"] = key
        os.environ["TYPESAFE_API_KEY"] = key
        print(f"✓ Loaded UNSLOTH_API_KEY from Kaggle Secrets: {key[:6]}...")
    except Exception:
        pass

    try:
        token = user_secrets.get_secret("HF_TOKEN")
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = token
        print("✓ Loaded HF_TOKEN from Kaggle Secrets.")
    except Exception:
        pass
except Exception:
    pass

print(f"✓ Environment configured in Keyless mode on port {PORT}.")

✓ Environment configured in Keyless mode on port 8000.


## 📦 2. Install Dependencies
Installs `laya[serve]`, `typesafe-sdk`, and telemetry utilities.

In [9]:
# Fast, conflict-free installation for Kaggle Linux T4
!pip install --no-warn-script-location --upgrade pip
!pip install --no-warn-script-location \
    "laya[serve]>=0.1.0" \
    "typesafe-sdk>=0.1.0" \
    requests tabulate numpy

## 🚀 3. Start Decision API Server in Background (Port 8000)
Launches the server on port `8000` using Python `subprocess.Popen`.

*Note:* Port `8888` is occupied by Kaggle's internal Jupyter web server. Port `8000` is completely free and is Laya's standard native port.

In [10]:
import os
import sys
import time
import subprocess

PORT = 8000

# 1. Clean up any previous server instance
try:
    subprocess.run(["pkill", "-9", "-f", "laya.serve"], stderr=subprocess.DEVNULL)
except Exception:
    pass

# 2. Explicit server environment variables
server_env = os.environ.copy()
server_env["LAYA_HOST"] = "127.0.0.1"
server_env["LAYA_PORT"] = str(PORT)
server_env["LAYA_MODELS"] = "multilingual"
server_env["LAYA_DEVICE"] = "cuda"

# 3. Spawn background daemon
log_file = open("unsloth_server.log", "w")
process = subprocess.Popen(
    [sys.executable, "-m", "laya.serve"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=server_env,
    start_new_session=True
)

print(f"✓ Decision API Server spawned with PID: {process.pid} on port {PORT}")
print("Waiting 15 seconds for model weights to cache onto GPU...")
time.sleep(15)

# 4. Verify process is running
if process.poll() is None:
    print(f"✓ Process {process.pid} is actively running and listening on port {PORT}!")
else:
    print(f"⚠ Process exited early with returncode {process.returncode}. Log contents:")
    with open("unsloth_server.log", "r") as f:
        print(f.read())

✓ Decision API Server spawned with PID: 303 on port 8000
Waiting 15 seconds for model weights to cache onto GPU...
✓ Process 303 is actively running and listening on port 8000!


## 🔬 4. Validation, Italian Test Ticket & 50-Call Speed Benchmark
1. **Health Verification:** Verifies `http://127.0.0.1:8000/health` returns `200 OK` JSON.
2. **Functional Test:** Evaluates an Italian support ticket with structured questions (`choice` and `noul`).
3. **50-Call Benchmark:** Tests 50 randomized **~200-word contexts**, measuring round-trip latency (RTT) and statistical averages on the T4 GPU.

In [11]:
import os
import time
import json
import random
import requests
import numpy as np
from tabulate import tabulate

PORT = 8000
BASE_URL = f"http://127.0.0.1:{PORT}"
SYSTEMONE_ENDPOINT = f"{BASE_URL}/v1/systemone"
API_KEY = os.environ.get("UNSLOTH_API_KEY", "sk-local-kaggle-token")
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

# ==========================================
# 1. STRICT HEALTH CHECK POLLING
# ==========================================
print(f"=== 1. Checking Server Status on Port {PORT} ===")

online = False
for attempt in range(15):
    try:
        res = requests.get(f"{BASE_URL}/health", timeout=2)
        # Laya /health returns status 200 and application/json
        if res.status_code == 200 and "application/json" in res.headers.get("content-type", ""):
            online = True
            print(f"✓ Server replied 200 OK: {res.json()}")
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(2)

if not online:
    print(f"\n⚠ Server on port {PORT} not ready. Printing unsloth_server.log:")
    if os.path.exists("unsloth_server.log"):
        with open("unsloth_server.log", "r") as f:
            print(f.read())
    raise RuntimeError(f"Server did not start cleanly on port {PORT}.")

print(f"Server Status: ✓ ONLINE on port {PORT}\n")

# =======================================================
# 2. FUNCTIONAL TEST: ITALIAN CUSTOMER SUPPORT TICKET
# =======================================================
print("=== 2. Functional Single-Call Test (Italian Ticket) ===")
sample_ticket = (
    "Gentile Assistenza Tecnica, sono bloccato da stamattina: ogni volta che tento di completare "
    "il pagamento con carta di credito sul vostro checkout aziendale, ricevo un errore 502 Bad Gateway "
    "e il carrello si svuota. La mia banca mi ha già scalato l'importo di 450€ ma l'ordine risulta inesistente. "
    "Ho urgenza assoluta di ricevere la fattura per chiudere il bilancio societario entro oggi pomeriggio."
)

decision_payload = {
    "model": "laya",
    "state": sample_ticket,
    "questions": {
        "is_urgent": {
            "type": "noul",
            "instructions": "Il ticket esprime un blocco operativo critico o urgenza finanziaria immediata?"
        },
        "department": {
            "type": "choice",
            "instructions": "Quale reparto aziendale deve gestire questa richiesta?",
            "criteria": {
                "billing_and_payments": "Problemi di fatturazione, transazioni rifiutate, addebiti duplicati o rimborsi.",
                "technical_bug": "Crash di sistema, errori HTTP 5xx, bug nel codice del frontend o backend.",
                "account_access": "Problemi con reset password, autenticazione a due fattori o blocco account.",
                "general_inquiry": "Domande informative su prodotti, cataloghi o orari di apertura."
            }
        }
    }
}

t0 = time.perf_counter()
response = requests.post(SYSTEMONE_ENDPOINT, headers=HEADERS, json=decision_payload, timeout=10)
initial_latency = (time.perf_counter() - t0) * 1000

if response.status_code == 200:
    print(f"✓ Status: 200 OK | Round-Trip Latency: {initial_latency:.2f} ms")
    print(f"Decisions Returned:\n{json.dumps(response.json(), indent=2, ensure_ascii=False)}\n")
else:
    print(f"⚠ Server response ({response.status_code}): {response.text}\n")

# ====================================================================
# 3. 50-CALL SPEED & LATENCY BENCHMARK (~200 WORDS CONTEXT EACH)
# ====================================================================
print("=== 3. 50-Call Speed & Latency Benchmark (~200 Words Context) ===")

CONTEXT_POOL = [
    (
        "Incident Report - Production Database Replica Desynchronization. "
        "At 14:23 UTC, our automated monitoring triggers indicated a persistent lag spike exceeding 45 seconds "
        "on the primary Postgres read-replica cluster serving our European microservices. Subsequent telemetry "
        "revealed an unindexed nested query originating from the analytics ETL batch pipeline, consuming 98% "
        "of available IOPS on the storage volume. Connection pools across the Kubernetes ingress tier quickly reached "
        "their maximum thresholds, yielding intermittent connection timeouts and HTTP 504 gateway responses for "
        "end users attempting to render real-time accounting dashboards. Failover automation was momentarily delayed "
        "due to active WAL replication locks. The on-call Site Reliability Engineering squad intervened by manually "
        "terminating the rogue query PID, temporarily restricting analytical throughput, and spinning up two ephemeral "
        "read replicas via Terraform to offload immediate read pressure. Database cache hit ratios normalized to 99.1% "
        "within eight minutes. All downstream payment webhooks and asynchronous messaging queues have completed "
        "reconciliation without data loss. We are conducting a thorough retrospective on schema migration reviews."
    ),
    (
        "Escalated Customer Dispute - Enterprise Subscription Renewal Discrepancy. "
        "We are contacting your legal and accounts department regarding enterprise contract number AGR-9842. "
        "Our agreement clearly stipulated a fixed annual rate of fifteen thousand dollars including dedicated "
        "enterprise support SLA guarantees and unlimited API token allowances for up to fifty provisioned engineering seats. "
        "However, our automated accounting ledger reconciliation shows that on October first, your automated billing "
        "gateway executed three consecutive unauthorized credit card charges totaling over twenty-one thousand dollars, "
        "citing overage surcharges that contradict section four of our signed master services agreement. Furthermore, "
        "our team experienced an unannounced suspension of our production workspace access tokens this morning, "
        "completely halting our active continuous integration deployment pipelines across our regional staging clusters. "
        "We demand an immediate reversal of the unauthorized deductions, immediate restoration of our primary administrator "
        "credentials, and formal written confirmation from your account executive before close of business today. "
        "Continued delay will force us to initiate formal commercial arbitration under international trade rules."
    ),
    (
        "Cybersecurity Incident Report - Suspicious Credential Stuffing Activity. "
        "During our scheduled 24/7 Security Operations Center review, automated behavioral analytics flagged an anomalous "
        "influx of distributed authentication requests targeting our customer identity portal. Over eighty thousand failed "
        "login events were logged across a forty-minute window, originating from a rotating residential proxy network "
        "spanning multiple geographic jurisdictions. The user-agent signatures demonstrated coordinated headless browser "
        "automation attempting to exploit credential dumps acquired from third-party data breaches. Our Cloudflare web "
        "application firewall rules dynamically throttled traffic exceeding five requests per second per IP block, while "
        "adaptive rate-limiting challenges were applied to all unauthenticated login routes. Preliminary database access "
        "audits show zero successful unauthorized account takeovers. However, eleven privileged accounts were temporarily "
        "locked out as an algorithmic precaution following repeated consecutive failures. Impacted stakeholders were "
        "promptly notified via verified out-of-band SMS notifications requiring hardware-backed MFA verification to unlock "
        "their profiles. The security operations team continues monitoring perimeter honeypots for subsequent probing attempts."
    ),
    (
        "Telecommunications Backbone Outage & Fibre Cut Diagnostics. "
        "At approximately 03:15 CEST, an automated telemetry alert reported a total loss of signal along our redundant "
        "dark-fibre transit route connecting Frankfurt and Zurich. Optical time-domain reflectometer diagnostics verified "
        "a severe physical severance approximately forty-two kilometers south of Stuttgart, likely caused by uncoordinated "
        "civil excavation equipment operating near the rail corridor. BGP path convergence automatically re-routed latency-sensitive "
        "financial traffic through our secondary Paris-Geneva ring within four hundred milliseconds, avoiding widespread "
        "packet drop. However, overall line capacity across the southern segment is currently degraded by sixty percent, causing "
        "jitter spikes for cross-border low-latency trading participants. Emergency field technicians have arrived at the splice "
        "chamber with replacement multi-core fiber spools. Fusion splicing operations are underway with an estimated completion "
        "time of three hours. Auxiliary power generators at the amplification hut remain online and fully stable."
    )
]

BENCHMARK_QUESTIONS = [
    {
        "priority": {
            "type": "choice",
            "instructions": "Determine operational triage priority.",
            "criteria": {
                "critical": "Outage, security breach, or significant legal/financial loss.",
                "medium": "Degradation or dispute without total service outage.",
                "low": "Informational or minor update."
            }
        },
        "human_escalation": {
            "type": "noul",
            "instructions": "Does this issue require human manager escalation?"
        }
    },
    {
        "risk_level": {
            "type": "choice",
            "instructions": "Assess severity risk level.",
            "criteria": {
                "high": "High financial, security or network risk.",
                "moderate": "Moderate risk with backup mitigation in place.",
                "low": "Low or negligible risk."
            }
        },
        "legal_threat": {
            "type": "noul",
            "instructions": "Does this incident involve potential legal liability or contract breach?"
        }
    }
]

# Warm-up calls
print("Warming up GPU kernels with 2 un-timed calls...")
for _ in range(2):
    requests.post(
        SYSTEMONE_ENDPOINT, 
        headers=HEADERS, 
        json={"model": "laya", "state": CONTEXT_POOL[0], "questions": BENCHMARK_QUESTIONS[0]},
        timeout=5
    )

NUM_CALLS = 50
latencies = []
table_rows = []

print(f"Executing {NUM_CALLS} benchmark calls against local Laya engine on NVIDIA T4 GPU...\n")
total_bench_start = time.perf_counter()

for i in range(1, NUM_CALLS + 1):
    ctx = random.choice(CONTEXT_POOL)
    q = random.choice(BENCHMARK_QUESTIONS)
    payload = {
        "model": "laya",
        "state": ctx,
        "questions": q
    }
    
    t_start = time.perf_counter()
    try:
        resp = requests.post(SYSTEMONE_ENDPOINT, headers=HEADERS, json=payload, timeout=5)
        t_elapsed = (time.perf_counter() - t_start) * 1000
        
        if resp.status_code == 200:
            latencies.append(t_elapsed)
            if i <= 10 or i % 10 == 0 or i == NUM_CALLS:
                table_rows.append([f"Call #{i:02d}", len(ctx.split()), f"{t_elapsed:.2f} ms", "200 OK"])
        else:
            table_rows.append([f"Call #{i:02d}", len(ctx.split()), f"{t_elapsed:.2f} ms", f"Err {resp.status_code}"])
    except Exception as ex:
        table_rows.append([f"Call #{i:02d}", len(ctx.split()), "Timeout", "Failed"])

total_bench_time = time.perf_counter() - total_bench_start

# Print sample results table
print("Sample Calls Progress (first 10 calls + periodic milestones):")
print(tabulate(table_rows, headers=["Run ID", "Context Words", "Latency (RTT)", "HTTP Status"], tablefmt="github"))

# Statistical Summary across all 50 calls
if latencies:
    lat_arr = np.array(latencies)
    print("\n" + "="*58)
    print(f"  BENCHMARK SUMMARY ({len(latencies)}/50 CALLS) - NVIDIA T4 GPU")
    print("="*58)
    print(f"  Total Timed Requests:    {len(lat_arr)}")
    print(f"  Total Wall-Clock Time:   {total_bench_time:.2f} s")
    print(f"  Mean Latency (Average):  {np.mean(lat_arr):.2f} ms")
    print(f"  Std Deviation:           {np.std(lat_arr):.2f} ms")
    print(f"  Median (P50 Latency):    {np.median(lat_arr):.2f} ms")
    print(f"  90th Percentile (P90):   {np.percentile(lat_arr, 90):.2f} ms")
    print(f"  95th Percentile (P95):   {np.percentile(lat_arr, 95):.2f} ms")
    print(f"  99th Percentile (P99):   {np.percentile(lat_arr, 99):.2f} ms")
    print(f"  Min Latency:             {np.min(lat_arr):.2f} ms")
    print(f"  Max Latency:             {np.max(lat_arr):.2f} ms")
    print(f"  Effective Throughput:    {len(lat_arr) / total_bench_time:.2f} req/sec (QPS)")
    print("="*58)

=== 1. Checking Server Status on Port 8000 ===
✓ Server replied 200 OK: {'status': 'ok', 'loaded': ['multilingual'], 'revisions': {'multilingual': '7b928d828b7b0e022f929d9bd2e44165aa270148'}, 'device': 'cuda', 'device_is_preference': False, 'checkpoint_devices': {'multilingual': 'cuda'}, 'cpu_fallbacks': {'multilingual': {'count': 0, 'last_reason': None}}}
Server Status: ✓ ONLINE on port 8000

=== 2. Functional Single-Call Test (Italian Ticket) ===
✓ Status: 200 OK | Round-Trip Latency: 6912.01 ms
Decisions Returned:
{
  "model": "laya-rl-agent",
  "answers": {
    "is_urgent": {
      "type": "noul",
      "noul": 1.0,
      "confidence": 1.0,
      "answer_confidence": 1.0,
      "action": {
        "act_probability": 1.0
      }
    },
    "department": {
      "type": "choice",
      "choice": "billing_and_payments",
      "probabilities": {
        "billing_and_payments": 0.4196,
        "technical_bug": 0.3602,
        "account_access": 0.1515,
        "general_inquiry": 0.0688
 

## 🧩 5. Native TypeSafe SDK Example
Calling the same endpoint using `typesafe-sdk` with `base_url="http://127.0.0.1:8000"`.

In [12]:
try:
    from typesafe_sdk import TypeSafeClient, Choice, Noul

    with TypeSafeClient(base_url="http://127.0.0.1:8000", api_key=os.environ.get("UNSLOTH_API_KEY", "local")) as client:
        decision = client.system_one(
            state="Richiedo rimborso immediato per fattura duplicata.",
            questions={
                "is_refund": Noul(instructions="Richiede rimborso?"),
                "urgency": Choice(instructions="Urgenza", criteria={"alta": "Urgente", "normale": "Standard"})
            }
        )
        print("✓ SDK Result:", decision)
except Exception as ex:
    print(f"ℹ SDK Call info: {ex}")

✓ SDK Result: model='laya-rl-agent' usage=Usage(input_tokens=85, output_tokens=0) answers={'is_refund': NoulAnswer(type='noul', noul=0.8195), 'urgency': ChoiceAnswer(type='choice', choice='alta', confidence=0.0156, probabilities={'alta': 0.5734, 'normale': 0.4266})}
